# Origins of major VOCs

Placement in the ARG of the characteristic mutations of Alpha, Delta and Omicron (BA.1/BA.2)
reported in the literature, for the supplementary section "Origins of major VOCs". The lists of
characteristic mutations are the same as those highlighted in the Extended Data subgraphs
(`supp_pdf-saltational-subgraphs.ipynb`). Pango origin nodes are from
`data/pango_events_in_arg.csv` (`analysis_pango_events_in_arg.ipynb`).

In [1]:
import collections

import numpy as np
import pandas as pd
import tskit
import tszip
import sc2ts

pd.set_option("display.max_rows", 200)

In [2]:
ts = tszip.load("../data/sc2ts_viridian_v2.3.trees.tsz")
df_node = sc2ts.node_data(ts, inheritance_stats=False)
df_mut = sc2ts.mutation_data(ts, inheritance_stats=False)
origin = pd.read_csv("../data/pango_events_in_arg.csv").set_index("pango")["root"]
origin[["B.1.1.7", "B.1.617", "B.1.617.1", "B.1.617.2", "B.1.617.3", "BA.1", "BA.2"]]

pango
B.1.1.7        94150
B.1.617       167120
B.1.617.1     187705
B.1.617.2     167136
B.1.617.3     212197
BA.1         1149212
BA.2          871742
Name: root, dtype: int64

In [3]:
def parse_mutation(entry):
    """
    Return the list of (position, derived_state) for one entry in a list of characteristic
    mutations; deletions are split into one mutation per site. Insertions and ambiguous
    entries are skipped, as in supp_pdf-saltational-subgraphs.ipynb.
    """
    entry = entry.replace("-", "_").strip()
    if "ins" in entry or "(" in entry:
        return []
    if "," in entry:
        return [x for part in entry.split(",") for x in parse_mutation(part)]
    if "_" in entry:
        pos = [int(p) for p in entry.replace("del", "").strip().split("_")]
        return [(p, "-") for p in range(pos[0], pos[-1] + 1)]
    if entry.endswith("del"):
        return [(int(entry[:-3]), "-")]
    return [(int(entry[1:-1]), entry[-1])]


def ancestors(u, position):
    tree = ts.at(position)
    path = []
    while u != -1:
        path.append(u)
        u = tree.parent(u)
    return path


def placement(position, derived_state, root, top):
    """
    Where a mutation occurs relative to the lineage whose origin node is root, and whose
    branch extends up to (but not including) the node top: "branch" (between root and top),
    "above" (at or above top), "below" (strictly below root, with the number of occurrences),
    or "absent".
    """
    tree = ts.at(position)
    df = df_mut[(df_mut["position"] == position) & (df_mut["derived_state"] == derived_state)]
    path = ancestors(root, position)
    branch = path[: path.index(top)]
    nodes = set(df["node"])
    if nodes & set(branch):
        return "branch"
    if nodes & set(path):
        return "above"
    below = [u for u in nodes if u != root and tree.is_descendant(u, root)]
    if len(below) > 0:
        return f"below ({len(below)})"
    return "absent"


variant = tskit.Variant(ts)
# Genotypes are indexed by position in ts.samples(), not by node ID
sample_index = np.full(ts.num_nodes, -1)
sample_index[ts.samples()] = np.arange(ts.num_samples)
site_positions = set(ts.sites_position.astype(int))


def fraction_carrying(position, derived_state, samples):
    """Fraction of the given samples with the derived state in the ARG (NaN if no site)."""
    if position not in site_positions:
        return np.nan
    variant.decode(ts.site(position=position).id)
    states = np.array(variant.alleles)[variant.genotypes[sample_index[samples]]]
    return np.mean(states == derived_state)


def main_node(position, derived_state, samples):
    """The node carrying the mutation that is ancestral to the most of the given samples."""
    tree = ts.at(position, tracked_samples=samples)
    df = df_mut[(df_mut["position"] == position) & (df_mut["derived_state"] == derived_state)]
    if len(df) == 0:
        return -1
    counts = [tree.num_tracked_samples(u) for u in df["node"]]
    return df["node"].iloc[np.argmax(counts)]


def classify(entries, root, top):
    """
    Placement of each characteristic mutation relative to the lineage with the given origin node,
    the fraction of the samples below the origin node that carry it, and the node carrying it
    for the most of these samples.
    """
    samples = np.array(list(ts.at(1000).samples(root)))
    rows = []
    for entry in entries:
        muts = parse_mutation(entry)
        if len(muts) == 0:
            continue
        where = [placement(p, d, root, top) if p in site_positions else "no site" for p, d in muts]
        # For multi-site mutations, summarise using the first site
        p, d = muts[0]
        u = main_node(p, d, samples)
        rows.append({
            "mutation": entry,
            "placement": "/".join(sorted(set(where))),
            "fraction": np.round(np.min([fraction_carrying(p, d, samples) for p, d in muts]), 3),
            "main_node": u,
            "main_node_pango": df_node.loc[u, "pango"] if u >= 0 else "",
        })
    return pd.DataFrame(rows)


def mutations(u):
    df = df_mut[df_mut["node"] == u]
    return [f"{a}{p}{b}" for a, p, b in zip(df["inherited_state"], df["position"], df["derived_state"])]


def node_mutations(u):
    """Mutations on node u, flagging reversions (back to the state before the parent mutation)."""
    df = df_mut[df_mut["node"] == u].copy()
    df["reversion"] = [
        p != -1 and df_mut.loc[p, "inherited_state"] == d for p, d in zip(df["parent"], df["derived_state"])]
    df["mutation"] = df["inherited_state"] + df["position"].astype(str) + df["derived_state"]
    return df[["mutation", "position", "reversion"]]


def group_mutations(nodes):
    events = []
    for u in nodes:
        df = df_mut[df_mut["node"] == u].sort_values("position")
        last = None
        for p, d in zip(df["position"], df["derived_state"]):
            if d == "-" and last is not None and last[1] == "-" and last[0] == p - 1 and events[-1][0] == u:
                events[-1] = (u, events[-1][1], p)
            else:
                events.append((u, p, p if d == "-" else None))
            last = (p, d)
    return events


def describe_path(u, position, n):
    rows = []
    for v in ancestors(u, position)[:n]:
        muts = mutations(v)
        rows.append({
            "node": v,
            "pango": df_node.loc[v, "pango"],
            "sample_id": df_node.loc[v, "sample_id"],
            "date": df_node.loc[v, "date"].date(),
            "num_mutations": len(muts),
            "num_deletion_sites": sum(m.endswith("-") for m in muts),
        })
    return pd.DataFrame(rows)

## Alpha

Mutations from https://virological.org/t/563 (Rambaut et al. 2020).

In [4]:
alpha_muts = """C913T
C3267T
C5388A
C5986T
T6954C
11288_11296 del
C14676T
C15279T
T16176C
21765-21770 del
21991-21993 del
A23063T
C23271A
C23604A
C23709T
T24506G
G24914C
T26801C
C27972T
G28048T
A28111G
G28280C, A28281T, T28282A
C28977T""".split("\n")
len(alpha_muts)

23

In [5]:
alpha_root = origin["B.1.1.7"]
describe_path(alpha_root, 1000, 4)

,node,pango,sample_id,date,num_mutations,num_deletion_sites
0,94150,B.1.1.7,,2020-09-16,36,19
1,5396,B.1.1,,2020-01-28,3,0
2,118,B.1.1,,2020-01-28,3,0
3,68,B.1,,2020-01-28,1,0


In [6]:
events = group_mutations([alpha_root])
print(len(events), "mutation events on the Alpha origin node, of which",
      sum(e[2] is not None for e in events), "deletions")

21 mutation events on the Alpha origin node, of which 4 deletions


The branch above the Alpha origin node ends at its B.1.1-labelled parent.

In [7]:
alpha_parent = ancestors(alpha_root, 1000)[1]
df_alpha = classify(alpha_muts, alpha_root, alpha_parent)
print(df_alpha["placement"].value_counts())
df_alpha

placement
branch       20
below (1)     1
absent        1
above         1
Name: count, dtype: int64


,mutation,placement,fraction,main_node,main_node_pango
0,C913T,branch,1.000,94150,B.1.1.7
1,C3267T,below (1),1.000,50503,B.1.1.7
2,C5388A,branch,1.000,94150,B.1.1.7
3,C5986T,branch,1.000,94150,B.1.1.7
4,T6954C,branch,0.999,94150,B.1.1.7
5,11288_11296 del,branch,1.000,94150,B.1.1.7
6,C14676T,branch,1.000,94150,B.1.1.7
7,C15279T,branch,1.000,94150,B.1.1.7
8,T16176C,branch,1.000,94150,B.1.1.7
9,21765-21770 del,branch,0.993,94150,B.1.1.7


In [8]:
print("Mutations on the Alpha origin node:", mutations(alpha_root))

Mutations on the Alpha origin node: ['C913T', 'C5388A', 'C5986T', 'T6954C', 'T11288-', 'C11289-', 'T11290-', 'G11291-', 'G11292-', 'T11293-', 'T11294-', 'T11295-', 'T11296-', 'C14676T', 'C15279T', 'T16176C', 'T21765-', 'A21766-', 'C21767-', 'A21768-', 'T21769-', 'G21770-', 'T21991-', 'T21992-', 'A21993-', 'A23063T', 'C23271A', 'C23604A', 'C23709T', 'T24506G', 'G24914C', 'C27972T', 'G28048T', 'A28111G', 'A28271-', 'C28977T']


C3267T is on a child of the Alpha origin node; the number of Alpha samples (below the origin
node) that are not below it.

In [9]:
tree = ts.at(3267)
u = df_alpha.set_index("mutation").loc["C3267T", "main_node"]
print(df_node.loc[u, "pango"], tree.parent(u) == alpha_root,
      tree.num_samples(alpha_root) - tree.num_samples(u), "of", tree.num_samples(alpha_root), "lack it")

B.1.1.7 True 1 of 308756 lack it


Children of the Alpha parent node, and the samples below it that are not Alpha.

In [10]:
tree = ts.at(1000)
children = tree.children(alpha_parent)
non_alpha = [c for c in children if c != alpha_root]
print(len(non_alpha), "other children, with", sum(tree.num_samples(c) for c in non_alpha), "samples")
df_node.loc[non_alpha, ["pango", "sample_id", "is_sample", "num_mutations"]]

7 other children, with 14 samples


,pango,sample_id,is_sample,num_mutations
1395,B.1.1,,False,1
5232,B.1.1,ERR4413600,True,0
7122,B.1.1,ERR4460507,True,1
1900363,B.1.1,ERR4460235,True,0
1900913,B.1.1,ERR4436889,True,0
1901497,B.1.1,ERR4460993,True,0
1901503,B.1.1,ERR4461063,True,0


Reference allele at 26801 (listed as T26801C by Rambaut et al.)

In [11]:
ts.site(position=26801).ancestral_state

'C'

## Delta

Characteristic mutations of the Kappa (B.1.617.1) and Delta (B.1.617.2) constellations and
the Pango designation issue for B.1.617, and of Delta clades A-D and E from Stern et al.

In [12]:
df = pd.read_csv("../data/Delta_subgraph_mutations.csv", keep_default_na=False)
df = df.rename(columns={df.columns[0]: "mutation"}).set_index("mutation") != ""
kappa = df["B.1.617.1 (Scorpio)"] | df["B.1.617 (Pango issue)"]
delta = df["B.1.617.2 (Scorpio)"]
category = pd.Series("", index=df.index)
category[kappa & delta] = "shared"
category[kappa & ~delta] = "Kappa"
category[~kappa & delta] = "Delta"
category[~kappa & ~delta & df["B.1.617.2 (Stern A-D)"]] = "clade A-D"
category[~kappa & ~delta & df["B.1.617.2 (Stern E)"]] = "clade E"
# A20396G is listed for both Kappa and Delta clade E
also_clade_e = kappa & ~delta & df["B.1.617.2 (Stern E)"]
assert (category != "").all()
category.value_counts()

Kappa        9
clade E      9
Delta        9
shared       6
clade A-D    3
Name: count, dtype: int64

In [13]:
delta_root = origin["B.1.617.2"]
kappa_root = origin["B.1.617.1"]
b1617_3_root = origin["B.1.617.3"]
tree = ts.at(1000)
mrca = tree.mrca(delta_root, kappa_root)
print("MRCA of Delta and Kappa:", mrca, df_node.loc[mrca, "pango"])
print("MRCA of Delta and B.1.617.3:", tree.mrca(delta_root, b1617_3_root))
print("MRCA of Kappa and B.1.617.3:", tree.mrca(kappa_root, b1617_3_root))
for name, u in [("Delta", delta_root), ("Kappa", kappa_root), ("B.1.617.3", b1617_3_root)]:
    print(name)
    display(describe_path(u, 1000, 6))

MRCA of Delta and Kappa: 167137 B.1
MRCA of Delta and B.1.617.3: 167138
MRCA of Kappa and B.1.617.3: 167138
Delta


,node,pango,sample_id,date,num_mutations,num_deletion_sites
0,167136,B.1.617.2,,2020-01-28,6,0
1,167137,B.1,,2020-01-28,2,0
2,167138,B.1,,2020-01-28,17,6
3,3416903,B.1,,2020-01-28,1,1
4,68,B.1,,2020-01-28,1,0
5,12,B.1,SRR11597205,2020-01-28,2,0


Kappa


,node,pango,sample_id,date,num_mutations,num_deletion_sites
0,187705,B.1.617.1,,2020-10-02,4,0
1,167120,B.1.617,,2020-07-03,14,0
2,167137,B.1,,2020-01-28,2,0
3,167138,B.1,,2020-01-28,17,6
4,3416903,B.1,,2020-01-28,1,1
5,68,B.1,,2020-01-28,1,0


B.1.617.3


,node,pango,sample_id,date,num_mutations,num_deletion_sites
0,212197,B.1.617.3,,2020-10-14,9,0
1,167138,B.1,,2020-01-28,17,6
2,3416903,B.1,,2020-01-28,1,1
3,68,B.1,,2020-01-28,1,0
4,12,B.1,SRR11597205,2020-01-28,2,0
5,33,B,,2019-12-26,1,0


Mutations on the nodes around the origins of Delta, Kappa and B.1.617.3.

In [14]:
for u in [167138, 167137, delta_root, 167135, 199569, 167120, kappa_root, b1617_3_root]:
    df = node_mutations(u)
    print(u, df_node.loc[u, "pango"], len(df), "mutations")
    print("   ", " ".join(m + ("(rev)" if r else "") for m, r in zip(df["mutation"], df["reversion"])))

167138 B.1 17 mutations
    G210T C21618G G21987A A22029- G22030- T22031- T22032- C22033- A22034- T22917G G23012C C23604G C25469T T27638C G28881T G29402T G29742T
167137 B.1 2 mutations
    C5184T A20396G
167136 B.1.617.2 6 mutations
    C22995A C23012G(rev) T26767C C27752T C28253T A28461G
167135 B.1.617.2 9 mutations
    G15451A C16466T G20396A(rev) G28248- A28249- T28250- T28251- T28252- T28253-
199569 B.1.617.2 6 mutations
    C1191T C1267T G9203A T9678C A17496G A21987G(rev)
167120 B.1.617 14 mutations
    C3457T C4965T A11201G G17523T G21618C(rev) T21895C G22022A -22029A(rev) -22030G(rev) -22031T(rev) -22032T(rev) -22033C(rev) -22034A(rev) A24775T
187705 B.1.617.1 4 mutations
    G16852T T20401G C26681T T26767G
212197 B.1.617.3 9 mutations
    C835T C7296T C8208T G10634T C11322T C16293T T27384C C27970T C28472T


C5184T is on the MRCA of Delta and Kappa but carried by few Delta samples: the parent of the
Delta origin node at 5184, and reversions at 5184 below it.

In [15]:
tree = ts.at(5184)
print("Parent of Delta origin at 5184:", tree.parent(delta_root))
df = df_mut[(df_mut["position"] == 5184)]
df = df[[tree.is_descendant(u, delta_root) for u in df["node"]]]
df.assign(samples=[tree.num_samples(u) for u in df["node"]]).sort_values("samples", ascending=False).head(3)[
    ["node", "inherited_state", "derived_state", "samples"]]

Parent of Delta origin at 5184: 167137


,node,inherited_state,derived_state,samples
584792,288419,T,C,1241388
584917,466999,C,T,349
584881,459136,C,T,113


B.1.617.3 samples in the ARG

In [16]:
df_node[df_node["is_sample"] & (df_node["pango"] == "B.1.617.3")][["sample_id", "date"]].sort_values("date")

,sample_id,date
167117,SRR23537641,2021-03-01
211706,SRR14876826,2021-03-20
215913,ERR5656264,2021-03-22
231345,SRR16679170,2021-03-31
236526,ERR5702831,2021-04-02
253618,SRR15911016,2021-04-12
257766,ERR5772561,2021-04-15
260044,SRR25001947,2021-04-16
260045,SRR25001948,2021-04-16
2118000,ERR5854658,2021-04-17


Fraction of the samples below each of the Delta, Kappa and B.1.617.3 origin nodes carrying each
mutation, and the node on which the mutation is carried by most of the Delta samples.

In [17]:
def lineage_fractions(entries, roots):
    tree = ts.at(1000)
    samples = {name: np.array(list(tree.samples(u))) for name, u in roots.items()}
    rows = []
    for entry in entries:
        muts = [(p, d) for p, d in parse_mutation(entry) if p in site_positions]
        if len(muts) == 0:
            continue
        row = {"mutation": entry}
        for name, ss in samples.items():
            row[name] = np.round(np.min([fraction_carrying(p, d, ss) for p, d in muts]), 3)
        rows.append(row)
    return pd.DataFrame(rows).set_index("mutation")


df_frac = lineage_fractions(category.index, {
    "Delta": delta_root, "Kappa": kappa_root, "B.1.617.3": b1617_3_root})
df_frac.insert(0, "category", category)
df_frac.sort_values("category")

,category,Delta,Kappa,B.1.617.3
mutation,,,,
G29402T,Delta,1.000,1.000,1.000
A28461G,Delta,0.999,0.000,0.000
28271_28271del,Delta,1.000,1.000,1.000
C27752T,Delta,1.000,0.000,0.000
T26767C,Delta,1.000,0.000,0.000
C22995A,Delta,1.000,0.000,0.000
22029_22034del,Delta,0.996,0.007,1.000
C21618G,Delta,1.000,0.003,1.000
A24775T,Kappa,0.000,1.000,0.000


Placement of each class of mutations: shared mutations relative to the MRCA of Delta and Kappa,
Kappa and Delta mutations relative to their origin nodes (branch up to the MRCA), and the
Stern et al. clade mutations relative to the Delta origin node.

In [18]:
rows = []
for cat, root, top in [
    ("shared", mrca, ancestors(mrca, 1000)[-1]),
    ("Kappa", kappa_root, mrca),
    ("Delta", delta_root, mrca),
    ("clade A-D", delta_root, mrca),
    ("clade E", delta_root, mrca),
]:
    entries = list(category[category == cat].index)
    if cat == "clade E":
        entries += list(category[also_clade_e].index)
    d = classify(entries, root, top)
    d.insert(0, "category", cat)
    rows.append(d)
df_delta = pd.concat(rows)
display(df_delta.groupby(["category", "placement"]).size())
df_delta

category   placement                    
Delta      above                            4
           branch                           4
           no site                          1
Kappa      above                            2
           branch                           7
clade A-D  below (2)                        2
           below (3)/below (4)/below (7)    1
clade E    above                            2
           below (12)                       1
           below (15)                       1
           below (156)                      1
           below (19)                       1
           below (22)                       1
           below (390)                      1
           below (4)                        1
           branch                           1
shared     branch                           6
dtype: int64

,category,mutation,placement,fraction,main_node,main_node_pango
0,shared,G21987A,branch,0.963,167138,B.1
1,shared,T22917G,branch,1.000,167138,B.1
2,shared,C23604G,branch,1.000,167138,B.1
3,shared,C25469T,branch,1.000,167138,B.1
4,shared,T27638C,branch,0.996,167138,B.1
5,shared,G28881T,branch,1.000,167138,B.1
0,Kappa,C3457T,branch,1.000,167120,B.1.617
1,Kappa,C4965T,branch,1.000,167120,B.1.617
2,Kappa,A11201G,branch,1.000,167120,B.1.617
3,Kappa,G17523T,branch,1.000,167120,B.1.617


The mutations below the Delta origin node that define clades A-D and E: the nodes on which they
occur, and the number of samples below each.

In [19]:
def occurrences(entries, root):
    rows = []
    for entry in entries:
        for p, d in parse_mutation(entry):
            tree = ts.at(p)
            df = df_mut[(df_mut["position"] == p) & (df_mut["derived_state"] == d)]
            for u in df["node"]:
                if tree.is_descendant(u, root):
                    rows.append({"mutation": entry, "position": p, "node": u,
                                 "pango": df_node.loc[u, "pango"], "samples": tree.num_samples(u)})
    return pd.DataFrame(rows)


clade_entries = list(category[category.isin(["clade A-D", "clade E"]) | also_clade_e].index)
df_occ = occurrences(clade_entries, delta_root)
df_occ = df_occ.join(category.rename("category"), on="mutation")
tree = ts.at(1000)
df_occ["delta_child"] = [
    next((c for c in tree.children(delta_root) if tree.is_descendant(u, c)), -1) for u in df_occ["node"]]
df_occ.sort_values("samples", ascending=False).drop_duplicates("mutation")

,mutation,position,node,pango,samples,category,delta_child
867,C28253T,28253,167136,B.1.617.2,1307146,clade E,-1
764,C16466T,16466,167135,B.1.617.2,1303892,clade A-D,167135
762,G15451A,15451,167135,B.1.617.2,1303521,clade A-D,167135
845,28248_28253del,28250,167135,B.1.617.2,1303511,clade A-D,167135
391,C1267T,1267,234948,B.1.617.2,13157,clade E,167135
1,C1191T,1191,234948,B.1.617.2,13157,clade E,167135
743,T9678C,9678,199569,B.1.617.2,3636,clade E,199569
724,G9203A,9203,199569,B.1.617.2,3636,clade E,199569
766,A17496G,17496,199569,B.1.617.2,3635,clade E,199569
747,C11005A,11005,290900,B.1.617.2,2719,clade E,199569


In [20]:
tree = ts.at(1000)
print("Children of the Delta origin node:")
pd.DataFrame([{"node": c, "pango": df_node.loc[c, "pango"], "samples": tree.num_samples(c)}
              for c in tree.children(delta_root)]).sort_values("samples", ascending=False).head()

Children of the Delta origin node:


,node,pango,samples
0,167135,B.1.617.2,1303454
1,199569,B.1.617.2,3636


## Omicron BA.1 and BA.2

Mutations from https://github.com/cov-lineages/pango-designation/issues/361, excluding those
marked as B.1 (inherited from earlier lineages).

In [21]:
df = pd.read_csv("../data/Omicron_BA.1_BA.2_mutations.csv", dtype=str, keep_default_na=False)
df = df.rename(columns={df.columns[0]: "mutation"})
df = df[~df["notes"].str.contains("B.1")]
omicron = {p: list(df[df[p].str.lower() == "y"]["mutation"]) for p in ["BA.1", "BA.2", "B.1.1.529"]}
{p: len(v) for p, v in omicron.items()}

{'BA.1': 20, 'BA.2': 27, 'B.1.1.529': 34}

In [22]:
ba1_root = origin["BA.1"]
ba2_root = origin["BA.2"]
tree = ts.at(1000)
omicron_mrca = tree.mrca(ba1_root, ba2_root)
print("MRCA of BA.1 and BA.2:", omicron_mrca, df_node.loc[omicron_mrca, "pango"])
describe_path(omicron_mrca, 1000, 8)

MRCA of BA.1 and BA.2: 871743 B.1.1.529


,node,pango,sample_id,date,num_mutations,num_deletion_sites
0,871743,B.1.1.529,,2020-10-11,64,31
1,871744,B.1.1,,2020-01-28,1,0
2,5904,B.1.1,,2020-01-28,1,0
3,118,B.1.1,,2020-01-28,3,0
4,68,B.1,,2020-01-28,1,0
5,12,B.1,SRR11597205,2020-01-28,2,0
6,33,B,,2019-12-26,1,0
7,1,B,Wuhan/Hu-1/2019,2019-12-26,0,0


The Omicron branch runs from the MRCA up to the first ancestor that is not labelled B.1.1.529.

In [23]:
path = ancestors(omicron_mrca, 1000)
omicron_top = next(u for u in path[1:] if df_node.loc[u, "pango"] != "B.1.1.529")
branch = path[: path.index(omicron_top)]
branch_muts = [m for u in branch for m in mutations(u)]
print("Top of branch:", omicron_top, df_node.loc[omicron_top, "pango"])
print(len(branch), "nodes,", len(branch_muts), "mutations, of which",
      sum(m.endswith("-") for m in branch_muts), "deletion sites")

Top of branch: 871744 B.1.1
1 nodes, 64 mutations, of which 31 deletion sites


Mutations on the Omicron branch, grouping adjacent deletion sites on the same node into one
deletion.

In [24]:
events = group_mutations(branch)
print(len(events), "mutation events, of which", sum(e[2] is not None for e in events), "deletions")
[(u, p, e) for u, p, e in events if e is not None]

38 mutation events, of which 5 deletions


[(871743, 11288, 11291),
 (871743, 21765, 21770),
 (871743, 21987, 21995),
 (871743, 22194, 22196),
 (871743, 28362, 28370)]

In [25]:
# The BA.2 origin node (871742) is the parent of node 1171677, which carries most BA.2-specific
# mutations and is ancestral to nearly all BA.2 samples (it is labelled as the BA.2 root in the
# Extended Data figure). We therefore classify BA.2 mutations relative to 1171677, so that
# the BA.2 branch comprises both nodes.
tree = ts.at(1000)
ba2_main = 1171677
assert tree.parent(ba2_main) == ba2_root
print(tree.num_samples(ba2_main), "of", tree.num_samples(ba2_root), "samples below the BA.2 origin are below", ba2_main)
rows = []
for p, root, top in [("B.1.1.529", omicron_mrca, omicron_top),
                     ("BA.1", ba1_root, omicron_mrca),
                     ("BA.2", ba2_main, omicron_mrca)]:
    d = classify(omicron[p], root, top)
    d.insert(0, "category", p)
    rows.append(d)
df_omicron = pd.concat(rows)
display(df_omicron.groupby(["category", "placement"]).size())
df_omicron

858979 of 859028 samples below the BA.2 origin are below 1171677


category   placement                  
B.1.1.529  above                           2
           below (3)/below (62)/branch     1
           branch                         30
BA.1       above                           6
           above/branch                    1
           absent                          1
           below (1)                       1
           branch                         10
BA.2       above                           1
           below (147)                     1
           below (49)                      1
           below (57)                      1
           branch                         23
dtype: int64

,category,mutation,placement,fraction,main_node,main_node_pango
0,B.1.1.529,C10029T,branch,1.000,871743,B.1.1.529
1,B.1.1.529,C10449A,branch,1.000,871743,B.1.1.529
2,B.1.1.529,11288_11296del,below (3)/below (62)/branch,0.665,871743,B.1.1.529
3,B.1.1.529,A18163G,branch,0.999,871743,B.1.1.529
4,B.1.1.529,G22578A,branch,1.000,871743,B.1.1.529
5,B.1.1.529,T22679C,branch,1.000,871743,B.1.1.529
6,B.1.1.529,C22686T,branch,1.000,871743,B.1.1.529
7,B.1.1.529,G22813T,branch,0.991,871743,B.1.1.529
8,B.1.1.529,T22882G,branch,0.979,871743,B.1.1.529
9,B.1.1.529,G22992A,above,1.000,5904,B.1.1


Fraction of BA.1 and BA.2 samples (below nodes 1149212 and 1171677) carrying each BA.1 and BA.2
characteristic mutation. Mutations placed on the shared B.1.1.529 node but specific to one
lineage must be reverted in the other.

In [26]:
df_frac_omicron = lineage_fractions(omicron["BA.1"] + omicron["BA.2"], {"BA.1": ba1_root, "BA.2": ba2_main})
df_frac_omicron.insert(0, "category", ["BA.1"] * len(omicron["BA.1"]) + ["BA.2"] * len(omicron["BA.2"])
                       if len(df_frac_omicron) == len(omicron["BA.1"]) + len(omicron["BA.2"]) else "")
df_frac_omicron

,category,BA.1,BA.2
mutation,,,
A2832G,,1.000,0.000
T5386G,,0.999,0.000
6513_6515del,,0.999,0.000
G8393A,,0.999,0.030
G11287T,,0.000,0.000
A11537G,,0.996,0.000
C15240T,,0.995,0.005
C21762T,,0.995,0.000
21765_21770del,,0.995,0.352


Mutations on the two BA.2 branch nodes. Reversions on node 1171677 undo BA.1-specific mutations
placed on the shared B.1.1.529 node; we count them as events, grouping adjacent deletion sites.

In [27]:
for u in [ba2_root, ba2_main]:
    df = node_mutations(u)
    print(u, len(df), "mutations,", df["reversion"].sum(), "reversions")
    print("   ", " ".join(m + ("(rev)" if r else "") for m, r in zip(df["mutation"], df["reversion"])))

df = node_mutations(ba2_main)
rev = df[df["reversion"]]["position"].values
num_events = 1 + np.sum(np.diff(rev) > 1)
print(num_events, "reversion events at", rev)

871742 16 mutations, 0 reversions
    T670G G4184A C4321T C9534T G10447A G11292- T11293- T11294- T11295- T11296- C12880T C15714T G22775A C26060T C26858T A29510C
1171677 43 mutations, 20 reversions
    C2790T C9344T A9424G C10198T C17410T C19955T A20055G C21618T T21633- A21634- C21635- C21636- C21637- C21638- C21639- T21640- G21641- T21762C(rev) -21765T(rev) -21766A(rev) -21767C(rev) -21768A(rev) -21769T(rev) -21770G(rev) T21846C(rev) -21987A -21988T(rev) -21989G(rev) -21990T(rev) -21991T(rev) -21992T(rev) -21993A(rev) -21994T(rev) -21995T(rev) -22194A(rev) -22195T(rev) -22196T(rev) T22200G A22688G A22898G(rev) G27382C A27383T T27384C
6 reversion events at [21762 21765 21766 21767 21768 21769 21770 21846 21988 21989 21990 21991
 21992 21993 21994 21995 22194 22195 22196 22898]


Path from the BA.2 node used as the BA.2 root in the Extended Data figure up to the BA.2
origin node.

In [28]:
describe_path(1171677, 1000, 6)

,node,pango,sample_id,date,num_mutations,num_deletion_sites
0,1171677,BA.2,,2021-10-19,43,9
1,871742,BA.2,,2021-03-22,16,5
2,871743,B.1.1.529,,2020-10-11,64,31
3,871744,B.1.1,,2020-01-28,1,0
4,5904,B.1.1,,2020-01-28,1,0
5,118,B.1.1,,2020-01-28,3,0


The BA.2 list gives A22786T (R408S), but the alignments have A22786C at this site.

In [29]:
display(classify(["A22786C"], ba2_main, omicron_mrca))
describe_path(871741, 1000, 3)

,mutation,placement,fraction,main_node,main_node_pango
0,A22786C,below (687),0.981,871741,BA.2


,node,pango,sample_id,date,num_mutations,num_deletion_sites
0,871741,BA.2,,2021-10-19,1,0
1,1171677,BA.2,,2021-10-19,43,9
2,871742,BA.2,,2021-03-22,16,5


G22992A and C26270T are on the two B.1.1-labelled nodes immediately above the B.1.1.529 node.
Samples below each, and below the B.1.1.529 node:

In [30]:
tree = ts.at(1000)
for u in [5904, 871744, omicron_mrca]:
    print(u, df_node.loc[u, "pango"], mutations(u) if u != omicron_mrca else "", tree.num_samples(u))

5904 B.1.1 ['G22992A'] 1303276
871744 B.1.1 ['C26270T'] 1294346
871743 B.1.1.529  1294334


Characteristic mutations of BA.1 and BA.2 that occur more than once below the origin node.

In [31]:
for p, root in [("BA.1", ba1_root), ("BA.2", ba2_root)]:
    d = occurrences(omicron[p], root)
    counts = d.groupby("mutation").size()
    print(p, dict(counts[counts > 1]))

BA.1 {'21765_21770del': np.int64(63), '21987_21995': np.int64(25), '22194_22196del': np.int64(202), '6513_6515del': np.int64(30), 'A26530G': np.int64(10), 'C15240T': np.int64(2), 'C21846T': np.int64(3), 'G22898A': np.int64(6)}


BA.2 {'21633_21641del': np.int64(14), 'A20055G': np.int64(2), 'A22688G': np.int64(6), 'A22786T': np.int64(148), 'A9424G': np.int64(2), 'C10198T': np.int64(20), 'C21618T': np.int64(9), 'C26858T': np.int64(85), 'C4321T': np.int64(81), 'C9534T': np.int64(2), 'C9866T': np.int64(49), 'G22775A': np.int64(2), 'G27382C, A27383T, T27384C': np.int64(13), 'T11296G': np.int64(57), 'T22200G': np.int64(11), 'T670G': np.int64(10)}


Deletion mutations at 11283-11296 on and below the Omicron branch.

In [32]:
df = df_mut[(df_mut["position"].between(11283, 11296)) & (df_mut["derived_state"] == "-")]
tree = ts.at(11290)
df = df[[tree.is_descendant(u, omicron_top) for u in df["node"]]]
df = df.assign(pango=df_node.loc[df["node"], "pango"].values,
               samples=[tree.num_samples(u) for u in df["node"]])
df.groupby(["node", "pango", "samples"])["position"].agg(["min", "max", "count"]).sort_values(
    "samples", ascending=False).head(10)

,,,min,max,count
node,pango,samples,,,
871743,B.1.1.529,1294274,11288,11291,4
871742,BA.2,860808,11292,11296,5
1149212,BA.1,433466,11283,11287,5
3416378,BA.1.15.1,26,11292,11294,3
1462777,BA.2,11,11283,11287,5
1099398,BA.1.1,7,11292,11294,3
1374159,BA.2,5,11283,11287,5
1158403,BA.2,5,11283,11287,5
3413770,BA.1.15,5,11292,11294,3
